In [26]:
import pandas as pd
import plotly.express as px

# 读取第二个工作表（索引为1）
df = pd.read_excel("附件：数据.xlsx", sheet_name=1, header=0)
# 提取地区列表和仓库距离数据
regions = df.columns[1:].tolist()  # 地区列名：石家庄、唐山、秦皇岛...
warehouse_a = df.iloc[0, 1:].values  # 仓库A（玉田）到各地区的距离
warehouse_b = df.iloc[1, 1:].values  # 仓库B（辛集）到各地区的距离
warehouse_c = df.iloc[2, 1:].values  # 仓库C（南宫）到各地区的距离
# 计算各仓库到各地区的最小距离
min_distances = [min(a, b, c) for a, b, c in zip(warehouse_a, warehouse_b, warehouse_c)]
# 示例：部分城市经纬度（需补充完整）
coordinates = {
    "石家庄": (38.0428, 114.5149),
    "唐山": (39.6309, 118.1803),
    "秦皇岛": (39.9354, 119.6005),
    "邯郸": (36.6256, 114.4909),
    "邢台": (37.0706, 114.5046),
    "保定": (38.8739, 115.4823),
    "张家口": (40.8111, 114.8864),
    "承德": (40.9756, 117.9392),
    "沧州": (38.3044, 116.8389),
    "廊坊": (39.5309, 116.6823),
    "衡水": (37.7354, 115.6761),
    "北京": (39.9042, 116.4074),
    "天津": (39.0842, 117.2010)
    # 其他城市需补充...
}

# 创建包含经纬度的DataFrame
map_df = pd.DataFrame({"地区": regions, "到最近仓库距离(km)": min_distances, "纬度": [coordinates[region] for region in regions], "经度": [coordinates[region] for region in regions]})
# 绘制地图

fig = px.scatter_geo(
    map_df,
    lat="纬度",
    lon="经度",
    hover_name="地区",
    size="到最近仓库距离(km)",
    color="到最近仓库距离(km)",
    scope="asia",
    center={
        "lat": 38,
        "lon": 115
    },  # 以河北省为中心
    title="华北地区到最近仓库的距离分布")

# 调整地图范围
# fig.update_geos(projection_scale=5, showcountries=False, showcoastlines=False, showland=True, landcolor="lightgray")

fig.show()


In [4]:
import pandas as pd
import folium

# 1. 读取附件2的数据（假设已加载为DataFrame）
# 数据示例（需根据实际Excel结构调整读取逻辑）：
data = {
    "地区": ["石家庄", "唐山", "秦皇岛", "邯郸", "邢台", "保定", "张家口", "承德", "沧州", "廊坊", "衡水", "北京", "天津"],
    "仓库A(玉田)": [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162, 246, 138, 376, 135.9, 123.2],
    "仓库B(辛集)": [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292, 271.7],
    "仓库C(南宫)": [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55, 335.1, 315.1]
}
df = pd.DataFrame(data).set_index("地区")

# 2. 确定每个地区的最近仓库及距离
nearest = df.apply(lambda row: row.idxmin(), axis=1)
distances = df.min(axis=1)

# 3. 手动定义地区经纬度（示例数据，需验证准确性）
locations = {
    "石家庄": (38.0422, 114.5086),
    "唐山": (39.6292, 118.1742),
    "秦皇岛": (39.9354, 119.6005),
    "邯郸": (36.6256, 114.4909),
    "邢台": (37.0706, 114.4753),
    "保定": (38.8735, 115.4645),
    "张家口": (40.8244, 114.8879),
    "承德": (40.9734, 117.9327),
    "沧州": (38.3047, 116.8388),
    "廊坊": (39.5191, 116.6838),
    "衡水": (37.7359, 115.6702),
    "北京": (39.9042, 116.4074),
    "天津": (39.0842, 117.2008)
}

# 4. 创建地图
m = folium.Map(location=[37.5, 115.0], zoom_start=6)  # 中心点设为河北省附近
colors = {"仓库A(玉田)": "red", "仓库B(辛集)": "blue", "仓库C(南宫)": "green"}

for region in df.index:
    lat, lng = locations.get(region, (0, 0))
    warehouse = nearest[region]
    distance = distances[region]
    
    folium.Marker(
        [lat, lng],
        popup=f"地区：{region}<br>最近仓库：{warehouse}<br>距离：{distance}公里",
        icon=folium.Icon(color=colors[warehouse], icon="info-sign")
    ).add_to(m)

# 保存为HTML文件
m.save("nearest_warehouse_map.html")
print("地图已生成：nearest_warehouse_map.html")

地图已生成：nearest_warehouse_map.html


In [5]:
import pandas as pd
from pyecharts.charts import Geo
from pyecharts import options as opts
from pyecharts.globals import GeoType, SymbolType

# 1. 数据准备（假设已从Excel读取数据）
# --------------------------------
# 读取附件2数据（模拟DataFrame）
data = {
    "地区": ["石家庄", "唐山", "秦皇岛", "邯郸", "邢台", "保定", "张家口", "承德", "沧州", "廊坊", "衡水", "北京", "天津"],
    "仓库A(玉田)": [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162, 246, 138, 376, 135.9, 123.2],
    "仓库B(辛集)": [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292, 271.7],
    "仓库C(南宫)": [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55, 335.1, 315.1]
}
df = pd.DataFrame(data).set_index("地区")

# 确定最近仓库及距离
nearest_warehouse = df.apply(lambda row: row.idxmin(), axis=1)
min_distances = df.min(axis=1).round(1)

# 2. 定义所有坐标（地区 + 仓库）
# --------------------------------
locations = {
    # 地区坐标（使用pyecharts内置城市名匹配）
    "石家庄": [114.52, 38.05],
    "唐山": [118.18, 39.63],
    "秦皇岛": [119.60, 39.94],
    "邯郸": [114.49, 36.62],
    "邢台": [114.48, 37.07],
    "保定": [115.46, 38.87],
    "张家口": [114.89, 40.82],
    "承德": [117.93, 40.97],
    "沧州": [116.84, 38.30],
    "廊坊": [116.68, 39.52],
    "衡水": [115.67, 37.73],
    "北京": [116.40, 39.90],
    "天津": [117.20, 39.08],
    # 仓库坐标（手动查询）
    "仓库A(玉田)": [117.73, 39.88],
    "仓库B(辛集)": [115.22, 37.94],
    "仓库C(南宫)": [115.38, 37.36]
}

# 3. 创建Geo地图
# --------------------------------
geo = (
    Geo(init_opts=opts.InitOpts(width="1200px", height="800px", theme="light"))
    .add_schema(
        maptype="河北",  # 聚焦河北省
        center=[116.4, 38.5],  # 地图中心
        zoom=6,  # 缩放级别
        itemstyle_opts=opts.ItemStyleOpts(color="#F0F0F0", border_color="#999")
    )
)

# 添加仓库位置（红色五角星标注）
for warehouse in ["仓库A(玉田)", "仓库B(辛集)", "仓库C(南宫)"]:
    geo.add_coordinate(warehouse, *locations[warehouse])
    geo.add(
        series_name="仓库",
        data_pair=[(warehouse, 0)],  # 数值设为0仅用于显示位置
        type_=GeoType.EFFECT_SCATTER,
        symbol_size=20,
        color="red",
        symbol=SymbolType.ROUND_RECT,
        label_opts=opts.LabelOpts(
            formatter="{b}", 
            position="right",
            font_size=12,
            color="black"
        )
    )

# 添加各地区及最近仓库信息（按仓库分颜色）
colors = {"仓库A(玉田)": "#FF6B6B", "仓库B(辛集)": "#4D8DF0", "仓库C(南宫)": "#34C759"}
for region in df.index:
    warehouse = nearest_warehouse[region]
    distance = min_distances[region]
    geo.add_coordinate(region, *locations[region])
    geo.add(
        series_name=warehouse,
        data_pair=[(region, distance)],
        type_=GeoType.SCATTER,
        symbol_size=12,
        color=colors[warehouse],
        label_opts=opts.LabelOpts(
            formatter=f"{region}\n{distance}公里",
            position="right",
            font_size=10
        )
    )

# 配置全局选项
geo.set_global_opts(
    title_opts=opts.TitleOpts(title="各地区到最近仓库的距离"),
    legend_opts=opts.LegendOpts(
        pos_top="5%",
        selected_mode="single",  # 可点击图例筛选
        textstyle_opts=opts.TextStyleOpts(color="#333")
    ),
    tooltip_opts=opts.TooltipOpts(trigger="item", formatter="{b}: {c}公里")
)

# 4. 保存为HTML并截图（需安装snapshot-phantomjs）
# --------------------------------
geo.render("warehouse_map.html")
print("地图已生成：warehouse_map.html")

# 如需保存为图片，使用以下代码（需提前安装snapshot-phantomjs）：
# from pyecharts.render import make_snapshot
# from snapshot_phantomjs import snapshot
# make_snapshot(snapshot, geo.render(), "warehouse_map.png")

地图已生成：warehouse_map.html


In [7]:
import pandas as pd
from pyecharts import options as opts
from pyecharts.charts import Map, Scatter
from pyecharts.globals import ChartType

# 仓库地理位置数据（经纬度）
warehouse_locations = {
    "仓库A(玉田)": {"name": "玉田", "latitude": 39.93, "longitude": 117.96},
    "仓库B(辛集)": {"name": "辛集", "latitude": 37.89, "longitude": 115.26},
    "仓库C(南宫)": {"name": "南宫", "latitude": 37.23, "longitude": 115.33}
}

# 读取 Excel 文件
excel_file = pd.ExcelFile('附件：数据.xlsx')
df = excel_file.parse('附件2')

# 提取地区名称和仓库数据
regions = df.columns[1:]
warehouses = df['单位:公里']

# 计算各地区到最近仓库的距离
nearest_distances = {}
nearest_warehouses = {}

for region in regions:
    min_distance = float('inf')
    nearest_warehouse = None
    
    for _, row in df.iterrows():
        warehouse = row['单位:公里']
        distance = row[region]
        if distance < min_distance:
            min_distance = distance
            nearest_warehouse = warehouse
    
    nearest_distances[region] = min_distance
    nearest_warehouses[region] = nearest_warehouse

# 创建地图数据
distance_data = list(nearest_distances.items())

# 创建地图
(
    Map()
    .add("到最近仓库的距离(km)", distance_data, "河北", is_map_symbol_show=False)
    .add(
        "仓库位置",
        [
            (warehouse, [warehouse_locations[warehouse]["longitude"], warehouse_locations[warehouse]["latitude"]])
            for warehouse in warehouse_locations
        ],
        "china",
        type_=ChartType.EFFECT_SCATTER,
        color="red",
        symbol_size=15,
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="各地区到最近仓库的距离及仓库位置"),
        visualmap_opts=opts.VisualMapOpts(
            min_=0,
            max_=600,
            range_text=["高", "低"],
            is_piecewise=True,
            pieces=[
                {"max": 100, "min": 0, "label": "0-100km"},
                {"max": 200, "min": 101, "label": "101-200km"},
                {"max": 300, "min": 201, "label": "201-300km"},
                {"max": 400, "min": 301, "label": "301-400km"},
                {"max": 500, "min": 401, "label": "401-500km"},
                {"max": 600, "min": 501, "label": "501-600km"},
            ],
        ),
        tooltip_opts=opts.TooltipOpts(trigger="item"),
        toolbox_opts=opts.ToolboxOpts(is_show=True),
        legend_opts=opts.LegendOpts(pos_top="5%"),
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=True, font_size=8),
    )
    .render("warehouse_distance_map_fixed.html")
)


TypeError: MapMixin.add() got an unexpected keyword argument 'type_'